<a href="https://colab.research.google.com/github/Priya-git2005/Gen-AI_CTTC/blob/main/Gen_AI_(Day4).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip uninstall -y protobuf

Found existing installation: protobuf 5.29.6
Uninstalling protobuf-5.29.6:
  Successfully uninstalled protobuf-5.29.6


In [2]:
!pip install -q protobuf==6.31.1 --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.1/321.1 kB 20.6 MB/s eta 0:00:00


In [4]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np

In [8]:
dataset,info = tfds.load("rock_paper_scissors",as_supervised=True,with_info=True)
train_data = dataset['train']
test_data = dataset['test']
print(info)

tfds.core.DatasetInfo(
    name='rock_paper_scissors',
    full_name='rock_paper_scissors/3.0.0',
    description="""
    Images of hands playing rock, paper, scissor game.
    """,
    homepage='http://laurencemoroney.com/rock-paper-scissors-dataset',
    data_dir='/root/tensorflow_datasets/rock_paper_scissors/3.0.0',
    file_format=tfrecord,
    download_size=219.53 MiB,
    dataset_size=219.23 MiB,
    features=FeaturesDict({
        'image': Image(shape=(300, 300, 3), dtype=uint8),
        'label': ClassLabel(shape=(), dtype=int64, num_classes=3),
    }),
    supervised_keys=('image', 'label'),
    disable_shuffling=False,
    nondeterministic_order=False,
    splits={
        'test': <SplitInfo num_examples=372, num_shards=1>,
        'train': <SplitInfo num_examples=2520, num_shards=2>,
    },
    citation="""@ONLINE {rps,
    author = "Laurence Moroney",
    title = "Rock, Paper, Scissors Dataset",
    month = "feb",
    year = "2019",
    url = "http://laurencemoroney.com/rock

In [10]:
#data split
IMG_S = 150
def preprocess(image,label):
  image = tf.image.resize(image,(IMG_S,IMG_S))
  image = image / 255 #normalize
  return image,label
# data mapping
train_data = train_data.map(preprocess).batch(32).shuffle(1000)
test_data = test_data.map(preprocess).batch(32)

In [15]:
# model
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32,(3,3),input_shape=(150,150,3),activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(64,(3,3),activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(128,(3,3),activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Flatten(),

    #hiiden layer
    tf.keras.layers.Dense(128,activation='relu'),

    #output
    tf.keras.layers.Dense(3,activation='softmax')
])

#Compile Layer
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [16]:
#training
model.fit(train_data,epochs=10,validation_data=test_data)

Epoch 1/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 19s 93ms/step - accuracy: 0.8218 - loss: 0.4496 - val_accuracy: 0.8522 - val_loss: 0.4199
Epoch 2/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.9992 - loss: 0.0050 - val_accuracy: 0.8441 - val_loss: 0.7450
Epoch 3/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 1.0000 - loss: 5.7035e-04 - val_accuracy: 0.8414 - val_loss: 0.9235
Epoch 4/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 1.0000 - loss: 1.0688e-04 - val_accuracy: 0.8414 - val_loss: 0.9815
Epoch 5/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 1.0000 - loss: 6.2092e-05 - val_accuracy: 0.8387 - val_loss: 1.0125
Epoch 6/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 1.0000 - loss: 4.7682e-05 - val_accuracy: 0.8360 - val_loss: 1.0180
Epoch 7/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 1.0000 - loss: 3.1513e-05 - val_accuracy: 0.8333 - val_loss: 1.0740
Epoch 8/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 1.0000 - loss: 2.0442e-05 -

In [17]:
#testing

import numpy as np
from tensorflow.keras.preprocessing import image